In [0]:
# Sample customers DataFrame
customers_data = [
    (1, "Alice", "Smith", "Acme Corp", "New York", "USA", "alice@acme.com", "acme.com"),
    (2, "Bob", "Jones", "Beta Inc", "London", "UK", "bob@beta.com", "beta.com"),
    (3, "Charlie", "Brown", "Gamma LLC", "Paris", "France", "charlie@gamma.com", "gamma.com"),
    (4, "Diana", "Prince", "Delta Ltd", "Berlin", "Germany", "diana@delta.com", "delta.com")
]
customers_columns = ["Customer_Id", "First_Name", "Last_Name", "Company", "City", "Country", "Email", "Website"]
customers_df = spark.createDataFrame(customers_data, customers_columns)
customers_df.createOrReplaceTempView("customers")

# Sample orders DataFrame
orders_data = [
    (101, 1, "2026-07-01"),
    (102, 2, "2026-07-02"),
    (103, 1, "2026-07-03")
]
orders_columns = ["Order_Id", "Customer_Id", "Order_Date"]
orders_df = spark.createDataFrame(orders_data, orders_columns)
orders_df.createOrReplaceTempView("orders")

display(customers_df)
display(orders_df)



# Customer who placed orders

In [0]:
%sql

select customer_id,first_name,last_name from customers where customer_id in (select customer_id from orders)


# Customers who have not placed any order


In [0]:
%sql

select customer_id,first_name,last_name from customers where customer_id not in (select customer_id from orders)


### Using join

In [0]:
%sql

select c.customer_id,c.first_name,c.last_name from customers c left join orders o on c.customer_id ==o.customer_id where o.customer_id is null


In [0]:
# Method 1: Using anti join
customers_no_orders_df = customers_df.join(orders_df, customers_df.Customer_Id == orders_df.Customer_Id, "left_anti")
display(customers_no_orders_df)

# Method 2: Using left join and filter nulls
customers_left_join_df = customers_df.join(orders_df, customers_df.Customer_Id == orders_df.Customer_Id, "left")
customers_no_orders_left_df = customers_left_join_df.filter(orders_df.Customer_Id.isNull())
display(customers_no_orders_left_df)

# Method 3: Using SQL NOT IN
customers_no_orders_sql = spark.sql("""
    select customer_id, first_name, last_name
    from customers
    where customer_id not in (select customer_id from orders)
""")
display(customers_no_orders_sql)

# Method 4: Using SQL LEFT JOIN and IS NULL
customers_no_orders_sql_join = spark.sql("""
    select c.customer_id, c.first_name, c.last_name
    from customers c
    left join orders o on c.customer_id = o.customer_id
    where o.customer_id is null
""")
display(customers_no_orders_sql_join)

In [0]:
from pyspark.sql import Row
from datetime import datetime, timedelta

# Sample customers data
customers_data = [
    Row(Customer_Id=1, First_Name="Alice", Last_Name="Smith", Email="alice@example.com"),
    Row(Customer_Id=2, First_Name="Bob", Last_Name="Jones", Email="bob@example.com"),
    Row(Customer_Id=3, First_Name="Charlie", Last_Name="Brown", Email="charlie@example.com"),
    Row(Customer_Id=4, First_Name="Diana", Last_Name="Prince", Email="diana@example.com")
]

customers_df = spark.createDataFrame(customers_data)
customers_df.createOrReplaceTempView('customers')
display(customers_df)

# Sample order details data
today = datetime(2026, 7, 21)
orders_data = [
    Row(Order_Id=101, Customer_Id=1, Order_Date=(today - timedelta(days=10)).strftime('%Y-%m-%d')),
    Row(Order_Id=102, Customer_Id=2, Order_Date=(today - timedelta(days=95)).strftime('%Y-%m-%d')),
    Row(Order_Id=103, Customer_Id=1, Order_Date=(today - timedelta(days=50)).strftime('%Y-%m-%d')),
    Row(Order_Id=104, Customer_Id=3, Order_Date=(today - timedelta(days=120)).strftime('%Y-%m-%d'))
]

orders_df = spark.createDataFrame(orders_data)
orders_df.createOrReplaceTempView('orders')
display(orders_df)


In [0]:
spark.sql("""select customer_id,first_name,last_name,current_date() as cnt_dt,date_sub(current_date(),100) as thirty_days_ago from customers 
          where customer_id  not in (
              select distinct customer_id from orders where order_date between date_sub(current_date(),100) and current_date())
              
          """).show(truncate=False)

In [0]:
spark.sql(""" select c.customer_id,c.first_name,c.last_name,o.order_date from customers c left join orders o on c.customer_id=o.customer_id  where o.order_date not between date_sub(current_date(),100) and current_date() """).show(truncate=False)

In [0]:
cust_df=customers_df.join(orders_df,"customer_id","left").filter(orders_df.Customer_Id.isNull())
cust_df.show(truncate=False)

## Customers who didnt place order since 100 days

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

odf=orders_df.select("Customer_Id","Order_Date").withColumn('cmp_date',date_sub(current_date(),100))\
    .filter(~col('order_date').between(col('cmp_date'),current_date()))
odf.show(truncate=False)


In [0]:
cdf=odf.join(customers_df,"Customer_Id",'left').show(truncate=False)